# 09 - XAI-Corrected Before/After Comparison

This notebook compares the original EfficientNetB0 fine-tuned model with the XAI-corrected EfficientNetB0 model from notebook 08.

The goal is not only to compare accuracy. The comparison asks:

- Did macro F1 or weak-class F1 improve?
- Did high-confidence errors reduce?
- Did Grad-CAM attention become more plausible?
- Is attention less background-driven and more focused on produce/decay evidence?

Run notebook 08 first. If the corrected model is missing, this notebook will stop early.


## 1. Project setup


In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(f"Could not find src/ from current directory: {NOTEBOOK_DIR}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


## 2. Imports and paths


In [ ]:
import importlib
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import Image as IPyImage, display

from src.config import CLASS_NAMES_PATH, FIGURES_DIR, GROUPED_SPLITS_DIR, MODELS_DIR, RANDOM_SEED, XAI_DIR

# Use grouped source-image splits to avoid offline-augmentation leakage.
SPLITS_DIR = GROUPED_SPLITS_DIR
from src.data.dataloaders import make_dataset_from_dataframe
import src.inference.xai as xai_module

xai_module = importlib.reload(xai_module)
explain_image_with_gradcam = xai_module.explain_image_with_gradcam
save_model_comparison_figure = xai_module.save_model_comparison_figure
select_gradcam_layer = xai_module.select_gradcam_layer

tf.keras.utils.set_random_seed(RANDOM_SEED)

BEFORE_MODEL_PATH = MODELS_DIR / "experiments" / "efficientnetb0_aug_oversampled_finetuned_wsl.keras"
AFTER_MODEL_PATH = MODELS_DIR / "experiments" / "efficientnetb0_xai_corrected.keras"
COMPARISON_DIR = XAI_DIR / "xai_corrected_before_after"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Before model:", BEFORE_MODEL_PATH)
print("After model:", AFTER_MODEL_PATH)


## 3. Load models and test split


In [ ]:
for path in [BEFORE_MODEL_PATH, AFTER_MODEL_PATH, CLASS_NAMES_PATH, SPLITS_DIR / "test.csv"]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}. Run notebook 08 first if the corrected model is missing.")

with open(CLASS_NAMES_PATH, "r", encoding="utf-8-sig") as f:
    class_names = json.load(f)

test_df = pd.read_csv(SPLITS_DIR / "test.csv")
class_to_index = {name: idx for idx, name in enumerate(class_names)}
if "class_index" not in test_df.columns:
    test_df["class_index"] = test_df["class_name"].map(class_to_index)

models_to_compare = {
    "EfficientNetB0 before": {
        "model": tf.keras.models.load_model(BEFORE_MODEL_PATH, compile=False),
        "prefix": "efficientnetb0",
        "preferred_layers": ["block6a_expand_conv", "top_conv"],
    },
    "EfficientNetB0 XAI-corrected": {
        "model": tf.keras.models.load_model(AFTER_MODEL_PATH, compile=False),
        "prefix": "efficientnetb0",
        "preferred_layers": ["block6a_expand_conv", "top_conv"],
    },
}

for label, config in models_to_compare.items():
    _, layer_name = select_gradcam_layer(config["model"], config["prefix"], config["preferred_layers"])
    config["gradcam_layer"] = layer_name
    print(label, "Grad-CAM layer:", layer_name)

print("Test rows:", len(test_df))


## 4. Metric and high-confidence error comparison


In [ ]:
def predict_audit_dataframe(model, label):
    test_ds = make_dataset_from_dataframe(test_df, class_to_index, training=False)
    probabilities = model.predict(test_ds, verbose=1)
    predicted_indices = np.argmax(probabilities, axis=1)
    confidences = np.max(probabilities, axis=1)

    df = test_df.copy().reset_index(drop=True)
    df["model"] = label
    df["predicted_index"] = predicted_indices
    df["predicted_class"] = [class_names[idx] for idx in predicted_indices]
    df["confidence"] = confidences
    df["correct"] = df["predicted_class"] == df["class_name"]
    df["high_confidence_error"] = (~df["correct"]) & (df["confidence"] >= 0.95)
    return df


audit_frames = {
    label: predict_audit_dataframe(config["model"], label)
    for label, config in models_to_compare.items()
}

summary_rows = []
for label, df in audit_frames.items():
    report_path = FIGURES_DIR / (
        "efficientnetb0_aug_oversampled_finetuned_wsl_classification_report.csv"
        if label.endswith("before")
        else "efficientnetb0_xai_corrected_classification_report.csv"
    )
    report_df = pd.read_csv(report_path, index_col=0) if report_path.exists() else None
    summary_rows.append({
        "model": label,
        "accuracy_from_audit_pass": float(df["correct"].mean()),
        "macro_f1": float(report_df.loc["macro avg", "f1-score"]) if report_df is not None else np.nan,
        "weighted_f1": float(report_df.loc["weighted avg", "f1-score"]) if report_df is not None else np.nan,
        "total_errors": int((~df["correct"]).sum()),
        "high_confidence_errors": int(df["high_confidence_error"].sum()),
        "mean_confidence_on_errors": float(df.loc[~df["correct"], "confidence"].mean()),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(COMPARISON_DIR / "metric_and_risk_summary.csv", index=False)
display(summary_df)


## 5. Weak rotten class comparison


In [ ]:
weak_classes = [
    "Tomato__Rotten",
    "Bellpepper__Rotten",
    "Potato__Rotten",
    "Pomegranate__Rotten",
    "Carrot__Rotten",
    "Guava__Rotten",
    "Jujube__Rotten",
]

weak_rows = []
for label in models_to_compare:
    report_path = FIGURES_DIR / (
        "efficientnetb0_aug_oversampled_finetuned_wsl_classification_report.csv"
        if label.endswith("before")
        else "efficientnetb0_xai_corrected_classification_report.csv"
    )
    if not report_path.exists():
        continue
    report_df = pd.read_csv(report_path, index_col=0)
    for class_name in weak_classes:
        if class_name in report_df.index:
            weak_rows.append({
                "model": label,
                "class_name": class_name,
                "precision": float(report_df.loc[class_name, "precision"]),
                "recall": float(report_df.loc[class_name, "recall"]),
                "f1_score": float(report_df.loc[class_name, "f1-score"]),
                "support": float(report_df.loc[class_name, "support"]),
            })

weak_df = pd.DataFrame(weak_rows)
weak_df.to_csv(COMPARISON_DIR / "weak_class_comparison.csv", index=False)
display(weak_df)


## 6. Same-image XAI comparison

The examples are selected from cases that are useful for audit: before-model errors, high-confidence errors, and weak rotten classes. This is intentional and not a random sample.


In [ ]:
before_df = audit_frames["EfficientNetB0 before"]
after_df = audit_frames["EfficientNetB0 XAI-corrected"]

shared = before_df[["image_path", "class_name", "predicted_class", "confidence", "correct", "high_confidence_error"]].merge(
    after_df[["image_path", "predicted_class", "confidence", "correct", "high_confidence_error"]],
    on="image_path",
    suffixes=("_before", "_after"),
)

candidate_parts = [
    shared[shared["high_confidence_error_before"]].sort_values("confidence_before", ascending=False).head(3),
    shared[(~shared["correct_before"]) & (shared["correct_after"])].sort_values("confidence_after", ascending=False).head(3),
    shared[shared["class_name"].isin(weak_classes)].sort_values("confidence_before", ascending=True).head(3),
]

comparison_examples_df = (
    pd.concat(candidate_parts, ignore_index=True)
    .drop_duplicates("image_path")
    .head(8)
    .reset_index(drop=True)
)
display(comparison_examples_df)


In [ ]:
def safe_filename(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text).strip("_")


figure_rows = []
for idx, row in comparison_examples_df.iterrows():
    explanations = []
    image_array = None

    for label, config in models_to_compare.items():
        explanation = explain_image_with_gradcam(
            model=config["model"],
            image_path=row["image_path"],
            class_names=class_names,
            target_layer_name=config["gradcam_layer"],
            base_model_name_prefix=config["prefix"],
            alpha=0.35,
        )
        explanation["model_label"] = label
        explanations.append(explanation)
        image_array = explanation["image_array"]

    figure_path = COMPARISON_DIR / f"{idx:02d}_{safe_filename(row['class_name'])}.png"
    save_model_comparison_figure(image_array, explanations, figure_path, true_class=row["class_name"])
    figure_rows.append({
        "image_path": row["image_path"],
        "true_class": row["class_name"],
        "figure_path": str(figure_path),
    })
    print(figure_path)
    display(IPyImage(filename=str(figure_path), width=1200))

figure_df = pd.DataFrame(figure_rows)
figure_df.to_csv(COMPARISON_DIR / "xai_before_after_figures.csv", index=False)
display(figure_df)


## 7. Interpretation template

Use this section to write the report conclusion after reviewing the figures.


In [ ]:
before_summary = summary_df[summary_df["model"] == "EfficientNetB0 before"].iloc[0]
after_summary = summary_df[summary_df["model"] == "EfficientNetB0 XAI-corrected"].iloc[0]

print("Report template:")
print(
    "XAI audit suggested possible shortcut learning and high-confidence errors. "
    "A corrected EfficientNetB0 experiment was trained with stronger geometric and color/noise augmentation. "
    f"Before correction, the model had {int(before_summary['high_confidence_errors'])} high-confidence errors. "
    f"After correction, it had {int(after_summary['high_confidence_errors'])} high-confidence errors. "
    "The final decision should consider both macro F1 and whether Grad-CAM attention became more aligned with produce/decay regions."
)
